# Objective: Analyse the efficiency and accuracy of autocomplete and autocorrect algorithms using NLP techniques. Implement and compare multiple approaches for text prediction and spelling correction on a real text dataset. 

### Tech Stack: Python, pandas, NLTK, pyspellchecker or textdistance, collections, matplotlib, Jupyter Notebook 

In [ ]:
# Data manipulation
import pandas as pd
import numpy as np

# Text processing
import nltk
import string
import re

# Collections
from collections import Counter, defaultdict

# Visualization
import matplotlib.pyplot as plt

# Spell correction
from spellchecker import SpellChecker

# Evaluation
from sklearn.metrics import confusion_matrix

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Download NLTK resources

nltk.download("punkt")
nltk.download("stopwords")
nltk.download("gutenberg")
nltk.download("words")

In [ ]:
# Load only first 10 million characters

with open("data/scraped_data.txt", "r", encoding="utf-8", errors="ignore") as file:
    corpus = file.read(10_000_000)

print("Characters Loaded :", len(corpus))

The dataset used in this project is the Wikipedia Text Corpus downloaded from Kaggle.

Since the complete dataset is extremely large (approximately 730 MB), only the first 10 million characters are loaded. This subset is sufficiently large for building autocomplete and autocorrect models while significantly reducing processing time and memory usage.

In [ ]:
print(corpus[:2000])

In [ ]:
characters=len(corpus)
words= corpus.split()
sentnces= nltk.sent_tokenize(corpus)

print("Total characters: ", characters)
print("Total words: ",len(words))
print("Total sentences: ",len(sentnces))

In [ ]:
import sys
print(f"corpus size in memory: {sys.getsizeof(corpus)/1024/1024:.2f} MB")

NLP Processing


In [ ]:
#converting to lower case
corpus=corpus.lower()
print(corpus[:500])

In [ ]:
#Remove numbers
corpus=re.sub(r"\d+"," ", corpus)
print(corpus[:500])

In [ ]:
#remove punctuation
corpus= corpus.translate(str.maketrans("","", string.punctuation))

In [ ]:
# Remove extra spaces

corpus = re.sub(r"\s+", " ", corpus).strip()
print(corpus[:500])

# Tokenization

Tokenization is the process of splitting a sentence into individual words called **tokens**.

Example:

Sentence:

Natural Language Processing is amazing.

Tokens:

["natural", "language", "processing", "is", "amazing"]

Tokenization is the foundation of almost every NLP application.

In [ ]:
from nltk.tokenize import word_tokenize

token= word_tokenize(corpus)
print("total number of tokens are: ", len(token))

print("first 50 tokens are:\n")
print(token[:50])

# Stopwords

Stopwords are very common words such as:

- the
- is
- are
- of
- in
- and

For many NLP tasks, removing stopwords helps reduce noise.

However, for **autocomplete**, stopwords are important because they help preserve natural sentence structure.

Therefore:

- A copy of the token list will retain stopwords for autocomplete.
- Another copy will remove stopwords for frequency analysis.

In [ ]:
from nltk.corpus import stopwords

stop_words= set(stopwords.words("english"))

#tokens for autocomplete (keep stopwords)
autocomplete_tokens= token.copy()

#tokens for EDA(remove stopwords)
clean_tokens= [word for word in token if word not in stop_words and word.isalpha()]

print("tokens with stopwords: ", len(autocomplete_tokens))
print("tokens without stopwords: ", len(clean_tokens))


# Vocabulary

The vocabulary is the collection of all unique words present in the corpus after preprocessing.

A larger vocabulary generally improves the diversity of autocomplete suggestions and increases the range of words that can be corrected by the autocorrect system.

In [ ]:
#count unique words
vocabulary= set(clean_tokens)
print("vocabulary size: ", len(vocabulary))

In [ ]:
print("sample vocabulary:\n")
print(list(vocabulary)[:50])

# Exploratory Data Analysis (EDA)

Before building the autocomplete model, it is useful to analyse the vocabulary and identify the most frequently occurring words in the corpus.

Word frequency analysis helps us understand the linguistic characteristics of the dataset and forms the basis of frequency-based language models.

In [ ]:
#count word frequencies
word_freq= Counter(clean_tokens)
print("Total unique words: ",len(word_freq))

#top 20 most frequent words
top_20= word_freq.most_common(20)
print(top_20)

In [ ]:
#fconverting to dataframe
freq_df=pd.DataFrame(
    top_20,
    columns=["Word","Frequency"]
)

print(freq_df)

# Top 20 Most Frequent Words

The following table displays the twenty most frequently occurring words after preprocessing.
Common words usually dominate large text corpora because they appear repeatedly across multiple documents.

In [ ]:
plt.figure(figsize=(12,6))

plt.bar(freq_df["Word"], freq_df["Frequency"])
plt.xticks(rotation=45)
plt.title("Top 20 most frquent words")
plt.xlabel("Words")
plt.ylabel("freuency")
plt.savefig('images/top20_most_freq.png')
plt.show()

The frequency distribution indicates that certain words occur significantly more often than others. These frequent words contribute heavily to statistical language models such as n-grams because higher occurrence generally corresponds to higher prediction probability.

In [ ]:
print("10 Least Frequent Words\n")
list(word_freq.items())[-10:]

In [ ]:
print('Average word length: ')
avg_lenght= np.mean([len(word) for word in clean_tokens])
print(round(avg_lenght,2))

In [ ]:
print("Longest word: ")
longest_word= max(vocabulary, key=len)
print(longest_word)
print(len(longest_word))

The exploratory analysis confirms that the corpus contains a rich vocabulary with sufficient word occurrences. The next step is to construct statistical language models that learn relationships between consecutive words for autocomplete prediction.

# Building the Bigram Language Model

A **bigram** is a sequence of two consecutive words.

For example, consider the sentence:

> Artificial intelligence is transforming healthcare.

The generated bigrams are:

- Artificial → intelligence
- intelligence → is
- is → transforming
- transforming → healthcare

The frequency of these word pairs forms the basis of a statistical autocomplete system.

In [ ]:
bigrams= list(zip(autocomplete_tokens[:-1], autocomplete_tokens[1:]))
print("Total bigrams: ", len(bigrams))

In [ ]:
#displaying first 10 bigrams
bigrams[:10]

In [ ]:
bigram_freq= Counter(bigrams)
print("Unique bigrams: ", len(bigram_freq))

In [ ]:
# Top 20 most frequent bigrams
bigram_freq.most_common(20)

# Building the Prediction Dictionary

To make autocomplete efficient, a dictionary is created where:

Key → Current word

Value → All possible next words along with their frequencies.

This allows predictions to be generated in constant time without searching through the entire corpus.

In [ ]:
autocomplete_dict= defaultdict(list)

for (first,second), freq in bigram_freq.items():
    autocomplete_dict[first].append((second,freq))

In [ ]:
# Sort each prediction list according to frequency

for word in autocomplete_dict:

    autocomplete_dict[word] = sorted(
        autocomplete_dict[word],
        key=lambda x: x[1],
        reverse=True
    )

In [ ]:
print("Sample Predictions for 'the':\n")
autocomplete_dict["the"][:10]

In [ ]:
print("Dictionary Size :", len(autocomplete_dict))

In [ ]:
def predict_bigram(word, top_n=3):
    word=word.lower()

    if word not in autocomplete_dict:
        return ["No prediction available"]

    predictions= autocomplete_dict[word][:top_n]

    return [next_word for next_word, _ in predictions]

In [ ]:
predict_bigram("the")

In [ ]:
test_words=[    
    "the",
    "machine",
    "artificial",
    "india",
    "science",
    "history",
    "computer",
    "language",
    "data",
    "university"
    ]

for word in test_words:
    print(f"\nInput:{word}")
    print("Prediction:", predict_bigram(word))

In [ ]:
sample_results= pd.DataFrame({
    "Input Word": test_words,
    "Top predictions": [predict_bigram(word)[0] for word in test_words]
})

sample_results

# Evaluation of the Autocomplete Model

To evaluate the effectiveness of the autocomplete system, a small ground truth dataset is created.

For each input word, one expected next word is defined based on common word associations in the corpus. The predicted words generated by the Bigram model are then compared with these expected outputs.

The evaluation metrics used are:

- **Precision** – Measures how many predicted words are relevant.
- **Recall** – Measures how many relevant words are successfully predicted.

In [ ]:
ground_truth = {
    "the": "first",
    "machine": "learning",
    "artificial": "intelligence",
    "india": "is",
    "science": "fiction",
    "history": "of",
    "computer": "science",
    "language": "is",
    "data": "analysis",
    "university": "of"
}

In [ ]:
evaluation_results=[]

for word in test_words:
    predictions= predict_bigram(word)
    expected= ground_truth[word]
    correct= expected in predictions

    evaluation_results.append([word, expected, predictions, correct])

evaluation_df= pd.DataFrame(
    evaluation_results,
    columns=[
        "Word",
        "Expected",
        "Predicted",
        "Correct"
    ]
)

evaluation_df

In [ ]:
true_positive = evaluation_df["Correct"].sum()
false_positive = len(evaluation_df) - true_positive
false_negative = false_positive

In [ ]:
precision = true_positive / (true_positive + false_positive)
recall = true_positive / (true_positive + false_negative)

print(f"Precision : {precision:.2f}")
print(f"Recall    : {recall:.2f}")

In [ ]:
plt.figure(figsize=(5,4))

plt.bar(
    ["Precision","Recall"],
    [precision, recall]
)

plt.ylim(0,1)

plt.title("Autocomplete Performance")

plt.ylabel("Score")
plt.savefig('images/Autocomp_performance.png')
plt.show()

## Autocorrect

# Part II – Autocorrect using Edit Distance

Autocomplete predicts the next word based on previously observed word sequences.

Autocorrect, on the other hand, detects spelling mistakes and suggests the most probable correct word.

Modern autocorrect systems combine multiple techniques such as edit distance, word frequency, language models and contextual understanding.

In this section, two different approaches are implemented:

1. **PySpellChecker** – A dictionary and edit-distance based spell correction library.
2. **Custom Levenshtein Distance** – A manually implemented edit-distance algorithm.

The performance of both approaches will then be compared.

In [ ]:
from spellchecker import SpellChecker
spell = SpellChecker()
print("SpellChecker Loaded Successfully!")

In [ ]:
#example
word="machne"
print("original word: ",word)
print("Corrected: ", spell.correction(word))

In [ ]:
#more examples
example_words=["languge", "sceinc","itellegence","algorthm","recieve"]

for word in example_words:
    print("Correted: ", spell.correction(word))

In [ ]:
test_words = {

    "machne":"machine",
    "langauge":"language",
    "scinece":"science",
    "algoritm":"algorithm",
    "recieve":"receive",
    "enviroment":"environment",
    "goverment":"government",
    "inteligence":"intelligence",
    "commitee":"committee",
    "seperate":"separate",
    "adress":"address",
    "occured":"occurred",
    "definately":"definitely",
    "beleive":"believe",
    "writting":"writing",
    "tommorrow":"tomorrow",
    "acheive":"achieve",
    "freind":"friend",
    "succesful":"successful",
    "neccessary":"necessary"
}

print("Total Test Words :", len(test_words))

In [ ]:
results=[]

for wrong, correct in test_words.items():
    predictions=spell.correction(wrong)

    results.append([wrong, correct, predictions, predictions==correct])

spell_df=pd.DataFrame(
    results,
    columns=[
        "Misspelled",
        "Expected",
        "Prediction",
        "Correct"
    ]
)

spell_df

In [ ]:
correct_predictions=spell_df["Correct"].sum()
accuracy= correct_predictions/len(spell_df)
print("Accuracy: ", accuracy)

The accuracy obtained above indicates the percentage of misspelled words that were correctly corrected by PySpellChecker.

Although the library performs well on common spelling mistakes, more challenging errors may still require contextual language models.

# Custom Levenshtein Distance Algorithm

Although PySpellChecker provides a ready-to-use spell correction system, it hides the internal implementation.

To better understand how spelling correction works, a custom Levenshtein Distance algorithm is implemented.

The algorithm measures the minimum number of edit operations required to transform one word into another.

The word with the smallest edit distance is selected as the predicted correction.

In [ ]:
def levenshtein_dist(s1, s2):

    if len(s1)<len(s2):
        return levenshtein_dist(s2,s1)

    if len(s2)==0:
        return len(s1)

    previous_row= list(range(len(s2)+1))

    for i,c1 in enumerate(s1):
        current_row= [i+1]

        for j, c2 in enumerate(s2):
            insertions= previous_row[j+1]+1
            deletions= current_row[j]+1
            substitutions=previous_row[j]+(c1!=c2)

            current_row.append(min(insertions, deletions, substitutions))

        previous_row=current_row

    return previous_row[-1]

In [ ]:
dictionary= list(vocabulary)
print("Dictionary size: ",len(dictionary))

In [ ]:
dictionary[:20]

In [ ]:
def custom_autocorrect(word):

    min_distance = float("inf")
    best_word = word

    for candidate in dictionary:

        distance = levenshtein_dist(
            word,
            candidate
        )

        if distance < min_distance:
            min_distance = distance
            best_word = candidate

    return best_word

In [ ]:
print(custom_autocorrect("machne"))

In [ ]:
custome_results=[]

for wrong, correct in test_words.items():
    predictions=custom_autocorrect(wrong)

    custome_results.append([wrong, correct, predictions, predictions==correct])

custom_df= pd.DataFrame(
    custome_results,
    columns=[
        "Wrong",
        "Expected",
        "Prediction",
        "Correct"
    ]
)

custom_df

The custom algorithm depends entirely on the quality of the vocabulary. If the corpus contains misspelled words, those errors become valid candidates.

In [ ]:
custom_correct = custom_df["Correct"].sum()
custom_accuracy = custom_correct / len(custom_df)
print(f"Custom Levenshtein Accuracy : {custom_accuracy:.2%}")

In [ ]:
comparison = pd.DataFrame({

    "Algorithm": [
        "PySpellChecker",
        "Custom Levenshtein"
    ],

    "Accuracy": [
        accuracy,
        custom_accuracy
    ]

})

comparison

In [ ]:
plt.figure(figsize=(6,4))

plt.bar(
    comparison["Algorithm"],
    comparison["Accuracy"]
)

plt.ylim(0,1)
plt.ylabel("Accuracy")
plt.title("Spell Correction Algorithm Comparison")
plt.savefig('images/accuracy_comp.png')
plt.show()

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

In [ ]:
y_true = [1] * len(custom_df)
y_pred = custom_df["Correct"].astype(int)

print("Accuracy :", accuracy_score(y_true, y_pred))
print("Precision :", precision_score(y_true, y_pred))
print("Recall :", recall_score(y_true, y_pred))

Since the evaluation dataset contains only misspelled words, there are no true negative examples. Consequently, precision remains high while recall reflects the proportion of words that were correctly corrected. A more comprehensive evaluation could include correctly spelled words to provide a more balanced confusion matrix.

In [ ]:
cm = confusion_matrix(y_true, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Incorrect", "Correct"]
)

disp.plot()

plt.title("Confusion Matrix - Custom Autocorrect")
plt.savefig('images/confusion_matrix.png')
plt.show()

# Completion of the Autocorrect Module

The autocorrect system has been successfully implemented using two different approaches:

1. PySpellChecker
2. Custom Levenshtein Distance

Both approaches were evaluated on a set of deliberately misspelled words, and their performance was compared using accuracy and additional evaluation metrics.

The following section discusses the limitations of the implemented system and compares it with production-grade autocomplete and autocorrect systems.

# Limitations of the Implemented System

Although the implemented autocomplete and autocorrect systems perform well for educational purposes, they have several limitations when compared to production-grade systems such as Google Keyboard, Microsoft SwiftKey, or Apple QuickType.

### Autocomplete Limitations

- The autocomplete model is based on **bigram frequencies**, which consider only the immediately preceding word.
- It cannot understand the overall context or meaning of a sentence.
- It does not adapt to individual user writing styles or frequently used phrases.
- Predictions are entirely dependent on the training corpus and may not generalize well to unseen text.
- The model cannot generate new phrases beyond those observed in the dataset.

### Autocorrect Limitations

- The custom Levenshtein algorithm compares every input word against the entire vocabulary, making it computationally expensive for large dictionaries.
- The algorithm relies solely on edit distance and does not consider word frequency or sentence context.
- Since the vocabulary is extracted from the corpus, any misspelled words present in the dataset may also become valid dictionary entries.
- PySpellChecker performs better due to its optimized dictionary and frequency-based ranking but still lacks contextual understanding.

### Comparison with Production Systems

Modern keyboard applications use advanced Natural Language Processing (NLP) techniques such as:

- Transformer-based language models
- Neural language models
- Context-aware prediction
- Personalized learning from user history
- Cloud-based language updates
- Semantic understanding of complete sentences

These techniques allow production systems to generate more accurate and contextually relevant suggestions than simple frequency-based or edit-distance-based approaches.